# Premier League Match Analysis and Prediction

Analysing four Premier League seasons (2021/22 to 2024/25) with SQL, then predicting match results with a Poisson model and comparing those predictions against Bet365's odds.

## Step 0: Setup check

This cell imports every library the project needs and prints its version. If it runs without an error, the environment is ready.

In [1]:
import sys                # Tells us which Python is running this notebook
import sqlite3            # SQL database, built into Python (no install needed)

import pandas as pd       # Tables of data (loading, cleaning, grouping)
import numpy as np        # Fast maths on arrays of numbers
import scipy              # Scientific maths (we'll use its Poisson distribution)
import matplotlib         # Charts

# This path should point inside the project's venv folder.
# If it does, the notebook is using the right environment.
print("Python running from:", sys.executable)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("sqlite:", sqlite3.sqlite_version)

Python running from: D:\Desktop\premier-league-match-predictor\venv\Scripts\python.exe
pandas: 3.1.1
numpy: 2.5.4
scipy: 1.18.1
matplotlib: 3.11.2
sqlite: 3.50.4


## Step 1: Load the data

Each season is a separate CSV file on football-data.co.uk. This cell downloads all four, labels every row with its season, and stacks them into one table. A Premier League season has 20 teams playing 380 matches, so we expect 380 rows per season and 1,520 in total.

In [2]:
# Each season's URL uses a short code: "2122" means 2021/22.
# This dictionary pairs each code with the label we want to use for that season.
seasons = {
    "2122": "2021/22",
    "2223": "2022/23",
    "2324": "2023/24",
    "2425": "2024/25",
}

season_tables = []  # An empty list to hold one table per season

for code, label in seasons.items():
    url = f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv"  # E0 = England's top division
    season_df = pd.read_csv(url).copy()  # Download the CSV into a table (.copy() tidies its memory layout)
    season_df["Season"] = label          # Add a column recording which season each row is from
    season_tables.append(season_df)      # Keep it until all four are loaded

# Stack the four tables on top of each other. ignore_index=True renumbers the rows
# 0, 1, 2... so that row numbers don't restart at 0 for each season.
raw_matches = pd.concat(season_tables, ignore_index=True)

# Check 1: the overall size, as (rows, columns)
print("Rows and columns:", raw_matches.shape)

# Check 2: each season should have exactly 380 matches
print("\nMatches per season:")
print(raw_matches["Season"].value_counts().sort_index())

# Check 3: look at the first few rows, using only the columns we plan to keep
columns_to_show = ["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]
raw_matches[columns_to_show].head()

Rows and columns: (1520, 133)

Matches per season:
Season
2021/22    380
2022/23    380
2023/24    380
2024/25    380
Name: count, dtype: int64


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,B365H,B365D,B365A
0,2021/22,13/08/2021,Brentford,Arsenal,2,0,H,4.00,3.40,1.95
1,2021/22,14/08/2021,Man United,Leeds,5,1,H,1.53,4.50,5.75
2,2021/22,14/08/2021,Burnley,Brighton,1,2,A,3.10,3.10,2.45
3,2021/22,14/08/2021,Chelsea,Crystal Palace,3,0,H,1.25,5.75,13.00
4,2021/22,14/08/2021,Everton,Southampton,3,1,H,1.90,3.50,4.00
